Simple GenAI app using Langchain

In [4]:
from dotenv import load_dotenv
import os
load_dotenv()
os.environ["USER_AGENT"] = "MyLangChainApp/1.0"

In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")
print(llm)

metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0', 'langchain-openai': '1.6.0'}} profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True} client=<openai.resources.chat.completions.completions.Completions object at 0x0000017554A8C980> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000017554A8D400> root_client=<openai.OpenAI object at 0x0000017553D7ECF0> root_async_cli

In [6]:
# Data Ingestion

from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://docs.langchain.com/langsmith/observability-concepts")

docs = loader.load()


C:\Users\anon1\AppData\Local\Temp\ipykernel_18196\1571671591.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


In [7]:
print(docs[0].page_content[:500]) # First 500 characters
print(docs[0].metadata)

Observability concepts - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationObservability conceptsOverviewTraceDebugObserveReferenc
{'source': 'https://docs.langchain.com/langsmith/observability-concepts', 'title': 'Observability concepts - Docs by LangChain', 'description': 'How LangSmith structures observability data as runs, traces, threads, and trajectories, and how to send traces.', 'language': 'en'}


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
print(f"Created {len(splits)} text chunks.")

Created 10 text chunks.


In [9]:
splits

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-concepts', 'title': 'Observability concepts - Docs by LangChain', 'description': 'How LangSmith structures observability data as runs, traces, threads, and trajectories, and how to send traces.', 'language': 'en'}, page_content="Observability concepts - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationObservability conceptsOverviewTraceDebugObserveReferenceQuickstartTutorialConceptsChatTracing setupIntegrationsManual instrumentationConfiguration & troubleshootingProject & environment settingsCost trackingUsage and billingAdvanced tracing tec

In [10]:
from langchain_openai.embeddings import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

from langchain_chroma import Chroma

db = Chroma.from_documents(
    splits, embedding=embeddings, persist_directory="./chroma_db"
)

In [11]:
db

In [12]:
db = Chroma(
    persist_directory="./chroma_db", 
    embedding_function=OpenAIEmbeddings()
)

query = "How LangSmith structures and visualizes data"
results = db.similarity_search(query, k=2)

In [13]:
print(results[0].page_content)

​How LangSmith structures and visualizes data
In LangSmith, every unit of work an agent performs, such as a model call, tool invocation, or information retrieval, is recorded as a run. The runs for a single operation are collected into a trace. You can link together traces from multi-turn sessions as a thread.
A trajectory is another way to structure and visualize that data. While a thread groups the traces of a session and keeps their nested structure, a trajectory flattens the entire session into an ordered list of messages that shows the path an agent took from start to finish.


In [14]:
## Retrieval Chain, Document Chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
     Answer the following question based only on the provided context
     <context>{context}</context>
    """)

document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n     Answer the following question based only on the provided context\n     <context>{context}</context>\n    '), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0', 'langchain-openai': '1.6.0'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_out

In [15]:
from langchain_core.documents import Document

result = document_chain.invoke(
    {
        "input": "Langchain has two usage limits: total traces and extended",
        "context": [
            Document(
                page_content="Langchain has two usage limits: total traces and extended traces. These correspond to the two metrices we've been tracking on our usage graph."
            )
        ],
    }
)

In [16]:
result

'What are the two usage limits for Langchain mentioned in the context? \n\nThe two usage limits for Langchain are total traces and extended traces.'

In [17]:
retriever = db.as_retriever()

from langchain_classic.chains import create_retrieval_chain

retrival_chain = create_retrieval_chain(retriever, document_chain)

In [18]:
retrival_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000175652CF4D0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n     Answer the following question based only on the provided context\n     <context>{context}</context>\n    '), additional_kwargs={})])
            | Ch

In [19]:
# Get Response from the LLM

response = retrival_chain.invoke({"input": "How LangSmith structures and visualizes data?"})

In [20]:
response

{'input': 'How LangSmith structures and visualizes data?',
 'context': [Document(id='2ddf2e47-43b1-43a4-b8bd-78957ec80bac', metadata={'title': 'Observability concepts - Docs by LangChain', 'language': 'en', 'description': 'How LangSmith structures observability data as runs, traces, threads, and trajectories, and how to send traces.', 'source': 'https://docs.langchain.com/langsmith/observability-concepts'}, page_content='\u200bHow LangSmith structures and visualizes data\nIn LangSmith, every unit of work an agent performs, such as a model call, tool invocation, or information retrieval, is recorded as a run. The runs for a single operation are collected into a trace. You can link together traces from multi-turn sessions as a thread.\nA trajectory is another way to structure and visualize that data. While a thread groups the traces of a session and keeps their nested structure, a trajectory flattens the entire session into an ordered list of messages that shows the path an agent took fr

In [21]:
response['answer']

"LangSmith structures and visualizes data using runs, traces, threads, and trajectories. Each unit of work performed by an agent (such as a model call or tool invocation) is recorded as a run. Runs related to a single operation are grouped into a trace. For multi-turn sessions, traces can be linked together to form a thread, which maintains the nested structure of the session. In contrast, a trajectory flattens the entire session into an ordered list of messages, providing a clear path of the agent's actions from start to finish. This framework allows for detailed recording, inspection, and analysis of an AI agent’s operations."

In [22]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template("""
You are a question-answering assistant.

Answer the question using only the provided context.
If the answer cannot be found in the context, say:
"I don't have enough information to answer this."

Context:
<context>
{context}
</context>

Question:
{input}
""")

retriever = db.as_retriever(search_kwargs={"k": 4})


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough(),
    }
    | prompt
    | llm
)

response = rag_chain.invoke(
    "How does LangSmith structure and visualize data?"
)

print(response.content)

LangSmith structures and visualizes data by recording every unit of work an agent performs as a run. The runs for a single operation are collected into a trace. Traces from multi-turn sessions can be linked together as a thread. Additionally, a trajectory is used to visualize data by flattening the entire session into an ordered list of messages, showing the path an agent took from start to finish.
